# Bootstrap — Intervalos de Confiança 95%
Estima a incerteza das métricas via reamostramento com reposição do conjunto de teste (2024).
Método: percentil (2.5% e 97.5% da distribuição bootstrap).
**Métricas:** Sensibilidade · AUPRC · ROC-AUC

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, average_precision_score, roc_auc_score
)

OUTPUT_MET  = '../../output/metricas'
OUTPUT_PLT  = '../../output/plots'
N_BOOTSTRAP = 2000
RANDOM_STATE = 42
os.makedirs(OUTPUT_PLT, exist_ok=True)

LABEL_MAP = {
    'logistic_regression_baseline':          'LR',
    'logistic_regression_baseline_tuned':    'LR (tuned)',
    'lightgbm_baseline':                     'LightGBM',
    'lightgbm_baseline_tuned':               'LightGBM (tuned)',
    'xgboost_baseline':                      'XGBoost',
    'xgboost_baseline_tuned':               'XGBoost (tuned)',
    'random_forest_baseline':                'Random Forest',
    'random_forest_baseline_tuned':          'Random Forest (tuned)',
    'decision_tree_baseline':                'Decision Tree',
    'decision_tree_baseline_tuned':          'Decision Tree (tuned)',
}

In [ ]:
predicoes = {}
for f in sorted(os.listdir(OUTPUT_MET)):
    if not f.endswith('_predicoes.parquet'):
        continue
    base = f.replace('_predicoes.parquet', '')
    if base not in LABEL_MAP:
        continue
    df   = pd.read_parquet(os.path.join(OUTPUT_MET, f))
    predicoes[base] = {
        'label':   LABEL_MAP[base],
        'y_true':  df['y_true'].values,
        'y_proba': df['y_proba'].values,
    }

print(f'{len(predicoes)} modelos carregados:')
for k, v in predicoes.items():
    print(f'  {v["label"]:<35} n={len(v["y_true"]):,}  óbitos={v["y_true"].sum():,}')

In [ ]:
from scipy.stats import bootstrap as scipy_bootstrap

def _sensibilidade(y_true, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, (y_proba >= 0.5).astype(int)).ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

ESTATISTICAS = {
    'sensibilidade': _sensibilidade,
    'auprc':         average_precision_score,
    'roc_auc':       roc_auc_score,
}

def bootstrap_metricas(y_true, y_proba, n=N_BOOTSTRAP, seed=RANDOM_STATE):
    resultados = {}
    for nome, func in ESTATISTICAS.items():
        res = scipy_bootstrap(
            (y_true, y_proba),
            statistic=func,
            n_resamples=n,
            paired=True,
            random_state=seed,
            method='percentile',
        )
        resultados[nome] = res.bootstrap_distribution
    return pd.DataFrame(resultados)

In [ ]:
# Carrega distribuições já computadas para evitar re-execução
dist_path = os.path.join(OUTPUT_MET, 'bootstrap_distribuicoes.parquet')
if os.path.exists(dist_path):
    df_dist_cache = pd.read_parquet(dist_path)
    labels_em_cache = set(df_dist_cache['modelo'].unique())
else:
    df_dist_cache = pd.DataFrame()
    labels_em_cache = set()

resultados_bs = {}

for arquivo, info in predicoes.items():
    label = info['label']
    if label in labels_em_cache:
        sub = df_dist_cache[df_dist_cache['modelo'] == label][['sensibilidade', 'auprc', 'roc_auc']]
        resultados_bs[arquivo] = sub.reset_index(drop=True)
        print(f'Carregado do cache: {label}')
    else:
        print(f'Bootstrap: {label}...', end=' ', flush=True)
        dist = bootstrap_metricas(info['y_true'], info['y_proba'])
        resultados_bs[arquivo] = dist
        print('OK')

print(f'\nConcluído — {N_BOOTSTRAP} iterações por modelo.')

In [ ]:
rows = []
for arquivo, dist in resultados_bs.items():
    info = predicoes[arquivo]
    y_true, y_proba = info['y_true'], info['y_proba']

    # estimativa pontual (conjunto completo)
    tn, fp, fn, tp = confusion_matrix(y_true, (y_proba >= 0.5).astype(int)).ravel()
    sens_pt  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    auprc_pt = average_precision_score(y_true, y_proba)
    roc_pt   = roc_auc_score(y_true, y_proba)

    for metrica, pt in [('sensibilidade', sens_pt), ('auprc', auprc_pt), ('roc_auc', roc_pt)]:
        lo = dist[metrica].quantile(0.025)
        hi = dist[metrica].quantile(0.975)
        rows.append({
            'modelo':   info['label'],
            'metrica':  metrica,
            'estimativa': round(pt, 4),
            'ic_low':   round(lo, 4),
            'ic_high':  round(hi, 4),
            'ic_width': round(hi - lo, 4),
        })

df_ic = pd.DataFrame(rows)

# tabela pivotada por métrica
for metrica in ['sensibilidade', 'auprc', 'roc_auc']:
    sub = (
        df_ic[df_ic['metrica'] == metrica]
        .sort_values('estimativa', ascending=False)
        [['modelo', 'estimativa', 'ic_low', 'ic_high', 'ic_width']]
        .reset_index(drop=True)
    )
    sub['IC 95%'] = sub.apply(lambda r: f"[{r['ic_low']:.4f}, {r['ic_high']:.4f}]", axis=1)
    print(f'\n=== {metrica.upper()} ===')
    display(sub[['modelo', 'estimativa', 'IC 95%', 'ic_width']])

In [ ]:
metricas_plot = [
    ('sensibilidade', '#C44E52', 'Sensibilidade'),
    ('auprc',         '#DD8452', 'AUPRC'),
    ('roc_auc',       '#4C72B0', 'ROC-AUC'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, max(5, len(predicoes) * 0.5)))

for ax, (metrica, cor, titulo) in zip(axes, metricas_plot):
    sub = (
        df_ic[df_ic['metrica'] == metrica]
        .sort_values('estimativa', ascending=True)
        .reset_index(drop=True)
    )
    y_pos = np.arange(len(sub))

    xerr_lo = sub['estimativa'] - sub['ic_low']
    xerr_hi = sub['ic_high']   - sub['estimativa']

    ax.errorbar(
        sub['estimativa'], y_pos,
        xerr=[xerr_lo, xerr_hi],
        fmt='o', color=cor, ecolor=cor, elinewidth=1.5,
        capsize=4, markersize=5,
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['modelo'], fontsize=8)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(
        max(0, sub['ic_low'].min() - 0.02),
        min(1, sub['ic_high'].max() + 0.02),
    )

plt.suptitle(f'Intervalos de Confiança 95% — Bootstrap ({N_BOOTSTRAP} iterações)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLT, 'bootstrap_forest_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Seleciona top 5 por AUPRC para visualizar a distribuição
top5 = (
    df_ic[df_ic['metrica'] == 'auprc']
    .sort_values('estimativa', ascending=False)
    .head(5)['modelo']
    .tolist()
)

arquivo_por_label = {v['label']: k for k, v in predicoes.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metrica in zip(axes, ['sensibilidade', 'auprc']):
    for label in top5:
        arq  = arquivo_por_label[label]
        dist = resultados_bs[arq][metrica]
        ax.hist(dist, bins=40, alpha=0.5, label=label, density=True)
    ax.set_title(f'Distribuição bootstrap — {metrica}', fontweight='bold')
    ax.set_xlabel(metrica)
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle(f'Top 5 modelos (por AUPRC) — distribuição das {N_BOOTSTRAP} amostras bootstrap',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLT, 'bootstrap_distribuicoes.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from itertools import combinations

for metrica in ['sensibilidade', 'auprc']:
    sub = df_ic[df_ic['metrica'] == metrica].set_index('modelo')
    modelos_ord = sub.sort_values('estimativa', ascending=False).index.tolist()

    print(f'\n=== Diferenças significativas — {metrica.upper()} ===')
    encontrou = False
    for m1, m2 in combinations(modelos_ord, 2):
        lo1, hi1 = sub.loc[m1, 'ic_low'], sub.loc[m1, 'ic_high']
        lo2, hi2 = sub.loc[m2, 'ic_low'], sub.loc[m2, 'ic_high']
        # ICs não se sobrepõem
        if hi1 < lo2 or hi2 < lo1:
            est1, est2 = sub.loc[m1, 'estimativa'], sub.loc[m2, 'estimativa']
            melhor = m1 if est1 > est2 else m2
            print(f'  {m1} vs {m2} → distinguíveis (melhor: {melhor})')
            encontrou = True
    if not encontrou:
        print('  Nenhum par com ICs não sobrepostos — diferenças não são estatisticamente distinguíveis.')

In [ ]:
os.makedirs(OUTPUT_MET, exist_ok=True)

# Resumo dos ICs
ic_path = os.path.join(OUTPUT_MET, 'bootstrap_ic95.csv')
df_ic.to_csv(ic_path, index=False)
print(f'ICs salvos: {ic_path}')

# Distribuições brutas (1 linha por iteração por modelo)
dist_rows = []
for arquivo, dist in resultados_bs.items():
    df_temp = dist.copy()
    df_temp['modelo'] = predicoes[arquivo]['label']
    dist_rows.append(df_temp)

df_dist = pd.concat(dist_rows, ignore_index=True)
dist_path = os.path.join(OUTPUT_MET, 'bootstrap_distribuicoes.parquet')
df_dist.to_parquet(dist_path, index=False)
print(f'Distribuições salvas: {dist_path} ({len(df_dist):,} linhas)')